# B2A Experiment — Serum Analysis
## Study 1 (One-Shot) & Study 2 (Multi-Step Agent)

**Category:** Facial Serum (8 fictional brands)  
**Model:** gpt-4o-mini | **Temperature:** 1.0  
**Design:** 8 conditions × 3 funnels × 4 input modes × 32 reps  
**DV:** `choseTarget` — whether the LLM selected the nudged product  
**Baseline (chance):** 12.5% (1/8 random)


## 0. Setup


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "sans-serif"

BASELINE = 1/8

CONDS = ["control","scarcity","social_proof_a","social_proof_b","urgency","authority_a","authority_b","price_anchoring"]
COND_LABELS = ["Control","Scarcity","Social\nProof A","Social\nProof B","Urgency","Authority\nA","Authority\nB","Price\nAnchoring"]
FUNNEL_ORDER = ["vague","moderate","specific"]
FUNNEL_LABELS = ["Vague\n(TOFU)","Moderate\n(MOFU)","Specific\n(BOFU)"]
MODE_ORDER = ["text_json","text_flat","html","screenshot"]
MODE_LABELS = ["JSON","Flat Text","HTML","Screenshot"]

S1_COLOR = "#5B9BD5"
S2_COLOR = "#ED7D31"

COND_COLORS = {
    "control":"#888888","scarcity":"#CC0C39","social_proof_a":"#232F3E",
    "social_proof_b":"#4A5568","urgency":"#B12704","authority_a":"#067D62",
    "authority_b":"#2D8B70","price_anchoring":"#D4380D",
}

def ci(hits, n, alpha=0.05):
    p = hits / n if n > 0 else 0
    lo, hi = proportion_confint(hits, n, alpha=alpha, method="wilson")
    return p, lo, hi


## 1. Data Loading


In [ ]:
DATA_DIR = Path("../results/260323_2")

s1_files = sorted(DATA_DIR.glob("study1/serum_experiment_*.jsonl"))
s1_raw = [json.loads(l) for f in s1_files for l in f.read_text().splitlines() if l.strip()]
s1 = pd.DataFrame(s1_raw) if s1_raw else pd.DataFrame()

s2_files = sorted(DATA_DIR.glob("study2/serum_experiment_*.jsonl"))
s2_raw = [json.loads(l) for f in s2_files for l in f.read_text().splitlines() if l.strip()]
s2 = pd.DataFrame(s2_raw) if s2_raw else pd.DataFrame()

for df in [s1, s2]:
    if len(df) == 0: continue
    if "funnel" not in df.columns: df["funnel"] = None
    for col in ["promptType", "agency"]:
        if col in df.columns: df["funnel"] = df["funnel"].fillna(df[col])
    drop_cols = [c for c in ["funnel","condition","inputMode","choseTarget"] if c in df.columns]
    df.dropna(subset=drop_cols, inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"Study 1: {len(s1)} trials" + (f" ({s1['choseTarget'].mean():.1%} hit rate)" if len(s1)>0 else ""))
print(f"Study 2: {len(s2)} trials" + (f" ({s2['choseTarget'].mean():.1%} hit rate)" if len(s2)>0 else ""))
print(f"Baseline: {BASELINE:.1%}")


## 2. Sanity Checks


In [ ]:
for label, df in [("Study 1", s1), ("Study 2", s2)]:
    if len(df) == 0:
        print(f"=== {label} === (no data)"); continue
    print(f"=== {label} ===")
    print(f"  Conditions: {df['condition'].nunique()} | Funnels: {df['funnel'].nunique()} | Modes: {df['inputMode'].nunique()}")
    print(f"  Reps/cell: {len(df) // (df['condition'].nunique() * df['funnel'].nunique() * df['inputMode'].nunique())}")
    ctrl = df[df['condition']=='control']['choseTarget'].mean()
    print(f"  Control hit rate: {ctrl:.1%} (expected: {BASELINE:.1%})\n")

if len(s1)>0:
    print("Target position distribution (S1):")
    print(s1['targetPosition'].value_counts().sort_index().to_string())


## 3. Hit Rate by Condition — S1 vs S2


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
s1r, s2r = [], []
s1_elo, s1_ehi, s2_elo, s2_ehi = [], [], [], []
s1_counts, s2_counts, s1_totals, s2_totals = [], [], [], []
for c in CONDS:
    s1_sub = s1[s1['condition']==c]; s2_sub = s2[s2['condition']==c]
    p1, lo1, hi1 = ci(s1_sub['choseTarget'].sum(), len(s1_sub))
    p2, lo2, hi2 = ci(s2_sub['choseTarget'].sum(), len(s2_sub))
    s1r.append(p1); s2r.append(p2)
    s1_elo.append(p1-lo1); s1_ehi.append(hi1-p1)
    s2_elo.append(p2-lo2); s2_ehi.append(hi2-p2)
    s1_counts.append(int(s1_sub['choseTarget'].sum())); s1_totals.append(len(s1_sub))
    s2_counts.append(int(s2_sub['choseTarget'].sum())); s2_totals.append(len(s2_sub))
x = np.arange(len(CONDS)); w = 0.35
ax.bar(x-w/2, s1r, w, label='Study 1 (One-Shot)', color=S1_COLOR, alpha=0.85)
ax.bar(x+w/2, s2r, w, label='Study 2 (Multi-Step)', color=S2_COLOR, alpha=0.85)
ax.errorbar(x-w/2, s1r, yerr=[s1_elo, s1_ehi], fmt='none', color='black', capsize=3, linewidth=1.2)
ax.errorbar(x+w/2, s2r, yerr=[s2_elo, s2_ehi], fmt='none', color='black', capsize=3, linewidth=1.2)
ax.axhline(y=BASELINE, color='gray', linestyle='--', alpha=0.5, label='Random (12.5%)')
ax.set_xticks(x); ax.set_xticklabels(COND_LABELS, fontsize=9)
ax.set_ylabel('Target Selection Rate'); ax.set_title('Target Selection Rate by Condition — Study 1 vs Study 2 (Serum)', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9); ax.set_ylim(0, max(max(s1r), max(s2r)) + 0.12)
for i in range(len(CONDS)):
    ax.text(x[i]-w/2, s1r[i]+s1_ehi[i]+0.01, f'{s1r[i]:.0%}\n({s1_counts[i]}/{s1_totals[i]})', ha='center', fontsize=7, color='#3a6d9e')
    ax.text(x[i]+w/2, s2r[i]+s2_ehi[i]+0.01, f'{s2r[i]:.0%}\n({s2_counts[i]}/{s2_totals[i]})', ha='center', fontsize=7, color='#c45a1a')
plt.tight_layout(); plt.savefig('../results/260323_2/fig01_s1_vs_s2.png', dpi=150, bbox_inches='tight'); plt.show()


## 4. Statistical Significance Table


In [ ]:
rows = []
for c in CONDS:
    for label, df in [("S1", s1), ("S2", s2)]:
        sub = df[df['condition']==c]; n = len(sub); hits = int(sub['choseTarget'].sum())
        p, lo, hi = ci(hits, n)
        if c != "control":
            ct = pd.crosstab(df[df['condition'].isin(['control', c])]['condition'], df[df['condition'].isin(['control', c])]['choseTarget'])
            chi2, pval, _, _ = stats.chi2_contingency(ct) if ct.shape == (2,2) else (0,1,0,None)
        else: chi2, pval = 0, 1
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        rows.append(dict(study=label, condition=c, n=n, hits=hits, rate=f"{p:.1%}", ci=f"[{lo:.1%},{hi:.1%}]", chi2=f"{chi2:.2f}", p=f"{pval:.4f}", sig=sig))
print(pd.DataFrame(rows).to_string(index=False))


## 5. Hit Rate by Funnel (TOFU / MOFU / BOFU)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (df, title, color) in zip(axes, [(s1, 'Study 1 (One-Shot)', S1_COLOR), (s2, 'Study 2 (Multi-Step)', S2_COLOR)]):
    x = np.arange(len(CONDS)); w = 0.25; funnel_colors = ['#5B9BD5','#70AD47','#FFC000']
    for i, (funnel, fl, fc) in enumerate(zip(FUNNEL_ORDER, FUNNEL_LABELS, funnel_colors)):
        rates, elo, ehi = [], [], []
        for c in CONDS:
            sub = df[(df['condition']==c) & (df['funnel']==funnel)]
            p, lo, hi = ci(sub['choseTarget'].sum(), len(sub))
            rates.append(p); elo.append(p-lo); ehi.append(hi-p)
        bars = ax.bar(x + (i-1)*w, rates, w, label=fl, color=fc, alpha=0.85)
        ax.errorbar(x + (i-1)*w, rates, yerr=[elo, ehi], fmt='none', color='black', capsize=2, linewidth=0.8)
        for j, bar in enumerate(bars):
            ax.text(bar.get_x()+bar.get_width()/2, rates[j]+ehi[j]+0.008, f'{rates[j]:.0%}', ha='center', fontsize=5.5)
    ax.axhline(y=BASELINE, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.set_xticks(x); ax.set_xticklabels(COND_LABELS, fontsize=8)
    ax.set_ylabel('Target Selection Rate'); ax.set_title(title, fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.legend(fontsize=8, loc='upper right')
plt.suptitle('Target Selection Rate by Funnel Stage × Condition', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('../results/260323_2/fig02_funnel.png', dpi=150, bbox_inches='tight'); plt.show()


## 6. Hit Rate by Input Mode


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
mode_colors = ['#5B9BD5','#ED7D31','#70AD47','#FFC000']
for ax, (df, title) in zip(axes, [(s1, 'Study 1 (One-Shot)'), (s2, 'Study 2 (Multi-Step)')]):
    x = np.arange(len(CONDS)); w = 0.2
    for i, (mode, ml, mc) in enumerate(zip(MODE_ORDER, MODE_LABELS, mode_colors)):
        rates, elo, ehi = [], [], []
        for c in CONDS:
            sub = df[(df['condition']==c) & (df['inputMode']==mode)]
            p, lo, hi = ci(sub['choseTarget'].sum(), len(sub))
            rates.append(p); elo.append(p-lo); ehi.append(hi-p)
        bars = ax.bar(x + (i-1.5)*w, rates, w, label=ml, color=mc, alpha=0.85)
        ax.errorbar(x + (i-1.5)*w, rates, yerr=[elo, ehi], fmt='none', color='black', capsize=2, linewidth=0.8)
        for j, bar in enumerate(bars):
            ax.text(bar.get_x()+bar.get_width()/2, rates[j]+ehi[j]+0.008, f'{rates[j]:.0%}', ha='center', fontsize=5)
    ax.axhline(y=BASELINE, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.set_xticks(x); ax.set_xticklabels(COND_LABELS, fontsize=8)
    ax.set_ylabel('Target Selection Rate'); ax.set_title(title, fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.legend(fontsize=8, loc='upper right')
plt.suptitle('Target Selection Rate by Input Mode × Condition', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('../results/260323_2/fig03_mode.png', dpi=150, bbox_inches='tight'); plt.show()


## 7. Heatmap — Condition × Funnel


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (df, title) in zip(axes, [(s1, 'Study 1'), (s2, 'Study 2')]):
    pivot = df.groupby(['condition','funnel'])['choseTarget'].mean().unstack()
    pivot = pivot.reindex(index=CONDS, columns=FUNNEL_ORDER)
    sns.heatmap(pivot, annot=pivot.map(lambda x: f'{x:.0%}'), fmt='', cmap='YlOrRd',
                vmin=0, vmax=0.7, ax=ax, cbar_kws={'format': mtick.PercentFormatter(1.0)},
                xticklabels=['Vague','Moderate','Specific'],
                yticklabels=[c.replace('_',' ').title() for c in CONDS])
    ax.set_title(title, fontsize=12); ax.set_xlabel('Funnel Stage'); ax.set_ylabel('')
plt.suptitle('Target Selection Rate Heatmap', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('../results/260323_2/fig04_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()


## 8. Statistical Tests
### 8a. Chi-squared: treatment pooled vs control
### 8b. Logistic regression


In [ ]:
for label, df in [("Study 1", s1), ("Study 2", s2)]:
    treat = df[df['condition'] != 'control']; ctrl = df[df['condition'] == 'control']
    table = pd.DataFrame({'hit': [ctrl['choseTarget'].sum(), treat['choseTarget'].sum()], 'miss': [(~ctrl['choseTarget']).sum(), (~treat['choseTarget']).sum()]}, index=['control','treatment'])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    print(f"{label}: χ²={chi2:.3f}, p={p:.6f}, dof={dof}")
    print(f"  Control: {ctrl['choseTarget'].mean():.1%} ({int(ctrl['choseTarget'].sum())}/{len(ctrl)})")
    print(f"  Treatment: {treat['choseTarget'].mean():.1%} ({int(treat['choseTarget'].sum())}/{len(treat)})\n")


In [ ]:
import statsmodels.formula.api as smf
for label, df in [("Study 1", s1), ("Study 2", s2)]:
    dm = df.copy(); dm['hit'] = dm['choseTarget'].astype(int)
    dm['condition'] = pd.Categorical(dm['condition'], categories=CONDS)
    dm['funnel'] = pd.Categorical(dm['funnel'], categories=FUNNEL_ORDER)
    dm['inputMode'] = pd.Categorical(dm['inputMode'], categories=MODE_ORDER)
    model = smf.logit("hit ~ C(condition, Treatment('control')) + C(funnel, Treatment('vague')) + C(inputMode, Treatment('text_json'))", data=dm)
    try:
        result = model.fit(disp=0)
        print(f"\n{'='*70}\n  {label} — Logistic Regression (ref: control, vague, text_json)\n{'='*70}")
        print(result.summary2().tables[1].to_string())
        print(f"\n  Pseudo R²: {result.prsquared:.4f}  |  AIC: {result.aic:.1f}  |  N: {result.nobs:.0f}")
    except Exception as e: print(f"{label}: regression failed — {e}")


## 9. Study 2: Agent Behavior Summary


In [ ]:
if len(s2) > 0 and 'toolCalls' in s2.columns:
    # Count tool calls from toolCalls array (more reliable than summary fields)
    def count_tool(tc_list, tool_name):
        if not isinstance(tc_list, list): return 0
        return sum(1 for t in tc_list if isinstance(t, dict) and t.get('tool') == tool_name)

    s2_beh = s2.copy()
    s2_beh['n_viewed'] = s2_beh['toolCalls'].apply(lambda x: count_tool(x, 'view_product'))
    s2_beh['n_reviews'] = s2_beh['toolCalls'].apply(lambda x: count_tool(x, 'read_reviews'))
    s2_beh['n_steps'] = s2_beh['toolCalls'].apply(lambda x: len(x) if isinstance(x, list) else 0)

    # Overall
    rows = [{'Mode': 'Overall',
            'Avg Tool Calls': s2_beh['n_steps'].mean(),
            'Products Viewed': s2_beh['n_viewed'].mean(),
            'Reviews Read': s2_beh['n_reviews'].mean(),
            'Badge Selection Rate': s2_beh['choseTarget'].mean()}]

    # By input mode
    for mode, ml in zip(MODE_ORDER, MODE_LABELS):
        sub = s2_beh[s2_beh['inputMode'] == mode]
        rows.append({'Mode': ml,
                     'Avg Tool Calls': sub['n_steps'].mean(),
                     'Products Viewed': sub['n_viewed'].mean(),
                     'Reviews Read': sub['n_reviews'].mean(),
                     'Badge Selection Rate': sub['choseTarget'].mean()})

    beh_df = pd.DataFrame(rows)
    beh_df['Badge Selection Rate'] = beh_df['Badge Selection Rate'].map(lambda x: f'{x:.1%}')
    for col in ['Avg Tool Calls','Products Viewed','Reviews Read']:
        beh_df[col] = beh_df[col].map(lambda x: f'{x:.1f}')
    print(beh_df.to_string(index=False))


## 9a. Study 2: Agent Step Depth


In [ ]:
if len(s2) > 0 and 'totalSteps' in s2.columns:
    step_stats = s2.groupby('condition').agg(mean_steps=('totalSteps','mean'), std_steps=('totalSteps','std')).reindex(CONDS)
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [COND_COLORS[c] for c in CONDS]
    ax.barh(range(len(CONDS)), step_stats['mean_steps'], color=colors, alpha=0.85)
    ax.errorbar(step_stats['mean_steps'], range(len(CONDS)), xerr=step_stats['std_steps'], fmt='none', color='black', capsize=3, linewidth=1)
    for i, (val, std) in enumerate(zip(step_stats['mean_steps'], step_stats['std_steps'])):
        ax.text(val + std + 0.3, i, f'{val:.1f}', va='center', fontsize=9, fontweight='bold')
    ax.set_yticks(range(len(CONDS))); ax.set_yticklabels([c.replace('_',' ').title() for c in CONDS], fontsize=9)
    ax.set_xlabel('Average Steps'); ax.set_title('Study 2 — Agent Exploration Depth by Condition', fontsize=12)
    ax.invert_yaxis(); plt.tight_layout()
    plt.savefig('../results/260323_2/fig05_agent_steps.png', dpi=150, bbox_inches='tight'); plt.show()


## 9b. User Journey Transitions (Sankey)


In [ ]:
if len(s2) > 0 and 'toolCalls' in s2.columns:
    try:
        import plotly.graph_objects as go
        HAS_PLOTLY = True
    except ImportError:
        HAS_PLOTLY = False

    # Extract transitions
    transitions = {}
    total_trials = 0
    for _, row in s2.iterrows():
        tc = row.get('toolCalls', [])
        if not isinstance(tc, list) or len(tc) == 0: continue
        total_trials += 1
        tools = [t['tool'] for t in tc if isinstance(t, dict) and 'tool' in t]
        for i in range(len(tools) - 1):
            pair = (tools[i], tools[i+1])
            transitions[pair] = transitions.get(pair, 0) + 1

    name_map = {'search': 'Search page', 'view_product': 'Product view',
                'read_reviews': 'Review page', 'select_product': 'Select product'}

    if HAS_PLOTLY:
        nodes = list(name_map.values())
        node_idx = {v: i for i, v in enumerate(nodes)}
        sources, targets, values = [], [], []
        for (src, tgt), count in transitions.items():
            sn, tn = name_map.get(src, src), name_map.get(tgt, tgt)
            if sn in node_idx and tn in node_idx:
                sources.append(node_idx[sn]); targets.append(node_idx[tn]); values.append(count)
        node_colors = ['rgba(136,132,216,0.8)','rgba(72,191,145,0.8)','rgba(59,130,246,0.8)','rgba(234,88,12,0.8)']
        link_colors = [node_colors[s].replace('0.8','0.25') for s in sources]
        fig = go.Figure(go.Sankey(
            node=dict(pad=20, thickness=25, label=nodes, color=node_colors),
            link=dict(source=sources, target=targets, value=values, color=link_colors)))
        fig.update_layout(title_text=f'User Journey Transitions (n={total_trials})', font_size=12, width=800, height=400)
        fig.show()
    else:
        # Fallback: transition matrix heatmap
        tools_list = ['search','read_reviews','view_product','select_product']
        tool_labels = ['Search','Reviews','Product View','Select']
        trans_matrix = np.zeros((4,4))
        for _, row in s2.iterrows():
            tc = row.get('toolCalls',[])
            if not isinstance(tc, list): continue
            tools = [t['tool'] for t in tc if isinstance(t, dict) and 'tool' in t]
            for i in range(len(tools)-1):
                if tools[i] in tools_list and tools[i+1] in tools_list:
                    trans_matrix[tools_list.index(tools[i])][tools_list.index(tools[i+1])] += 1
        row_sums = trans_matrix.sum(axis=1, keepdims=True); row_sums[row_sums==0] = 1
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(trans_matrix / row_sums, annot=trans_matrix.astype(int), fmt='d', cmap='Blues',
                    xticklabels=tool_labels, yticklabels=tool_labels, ax=ax)
        ax.set_xlabel('To'); ax.set_ylabel('From')
        ax.set_title(f'Study 2 — Tool Call Transition Matrix (n={len(s2)})', fontsize=12)
        plt.tight_layout(); plt.savefig('../results/260323_2/fig07_transitions.png', dpi=150, bbox_inches='tight'); plt.show()

    print('\nTop transitions:')
    for (src, tgt), count in sorted(transitions.items(), key=lambda x: -x[1])[:10]:
        print(f"  {name_map.get(src,src):20s} → {name_map.get(tgt,tgt):20s}: {count:5d} ({count/total_trials:.1%} of trials)")


## 9c. Exploration Depth vs Badge Selection Rate


In [ ]:
if len(s2) > 0 and 'totalSteps' in s2.columns:
    bin_labels = ['1-3', '4-6', '7-10', '11+']
    s2c = s2.copy()
    s2c['depth_bin'] = pd.cut(s2c['totalSteps'], bins=[0, 3, 6, 10, 999], labels=bin_labels)
    s2c['is_treatment'] = s2c['condition'] != 'control'

    # ── Main plot: all funnels ──
    fig, ax = plt.subplots(figsize=(10, 5))
    for group, color, label in [(False, S1_COLOR, 'Control'), (True, S2_COLOR, 'Treatment')]:
        sub = s2c[s2c['is_treatment'] == group]
        rates, elo, ehi, ns = [], [], [], []
        for bl in bin_labels:
            bsub = sub[sub['depth_bin'] == bl]; n = len(bsub); hits = int(bsub['choseTarget'].sum())
            p, lo, hi = ci(hits, n) if n > 0 else (0, 0, 0)
            rates.append(p); elo.append(p-lo); ehi.append(hi-p); ns.append(n)
        x = np.arange(len(bin_labels))
        ax.errorbar(x, rates, yerr=[elo, ehi], fmt='-o', color=color, capsize=4, linewidth=2, markersize=8, label=label)
        for i in range(len(bin_labels)):
            ax.text(x[i], rates[i]+ehi[i]+0.02, f'{rates[i]:.0%}\n(n={ns[i]})', ha='center', fontsize=8, color=color)
    ax.axhline(y=BASELINE, color='gray', linestyle='--', alpha=0.5, label='Random (12.5%)')
    ax.set_xticks(range(len(bin_labels))); ax.set_xticklabels(bin_labels)
    ax.set_xlabel('Exploration Depth (total steps)', fontsize=11)
    ax.set_ylabel('Badge Selection Rate', fontsize=11)
    ax.set_title('Exploration Depth and Badge Selection Rate (S2, all funnels)', fontsize=13)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(fontsize=10); ax.set_ylim(0, 0.5); ax.set_xlim(-0.3, len(bin_labels)-0.7)
    plt.tight_layout(); plt.savefig('../results/260323_2/fig08_depth_vs_selection.png', dpi=150, bbox_inches='tight'); plt.show()

    # ── By funnel ──
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    for ax, funnel, fl in zip(axes, FUNNEL_ORDER, ['Vague (TOFU)', 'Moderate (MOFU)', 'Specific (BOFU)']):
        sub_f = s2c[s2c['funnel'] == funnel]
        for group, color, label in [(False, S1_COLOR, 'Control'), (True, S2_COLOR, 'Treatment')]:
            sub = sub_f[sub_f['is_treatment'] == group]
            rates, elo, ehi, ns = [], [], [], []
            for bl in bin_labels:
                bsub = sub[sub['depth_bin']==bl]; n = len(bsub); hits = int(bsub['choseTarget'].sum())
                p, lo, hi = ci(hits, n) if n > 0 else (0, 0, 0)
                rates.append(p); elo.append(p-lo); ehi.append(hi-p); ns.append(n)
            ax.errorbar(range(len(bin_labels)), rates, yerr=[elo, ehi], fmt='-o', color=color, capsize=3, linewidth=1.5, markersize=6, label=label)
            for i in range(len(bin_labels)):
                if ns[i] > 0: ax.text(i, rates[i]+ehi[i]+0.015, f'{rates[i]:.0%}\n(n={ns[i]})', ha='center', fontsize=6, color=color)
        ax.axhline(y=BASELINE, color='gray', linestyle='--', alpha=0.5)
        ax.set_xticks(range(len(bin_labels))); ax.set_xticklabels(bin_labels)
        ax.set_xlabel('Exploration Depth'); ax.set_title(fl, fontsize=11)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        if ax == axes[0]: ax.set_ylabel('Badge Selection Rate'); ax.legend(fontsize=8)
    plt.suptitle('Exploration Depth vs Badge Selection Rate by Funnel', fontsize=13, y=1.02)
    plt.tight_layout(); plt.savefig('../results/260323_2/fig09_depth_by_funnel.png', dpi=150, bbox_inches='tight'); plt.show()


## 10. Reasoning Analysis — Badge Mention


In [ ]:
BADGE_KEYWORDS = {
    "scarcity": ["left in stock","order soon","limited","scarce"],
    "social_proof_a": ["best seller","#1","popular","top rated"],
    "social_proof_b": ["viewing","people","popular"],
    "urgency": ["deal ends","hurry","limited time","countdown"],
    "authority_a": ["recommended","dermatologist","expert","endorsed"],
    "authority_b": ["clinically","tested","certified","proven"],
    "price_anchoring": ["was","save","discount","deal","original price"],
}
mention_data = []
for label, df in [("Study 1", s1), ("Study 2", s2)]:
    for cond in CONDS:
        if cond == "control": continue
        hits = df[(df['condition']==cond) & (df['choseTarget']==True)]
        if len(hits) == 0: continue
        keywords = BADGE_KEYWORDS.get(cond, [])
        mentions = hits['reasoning'].apply(lambda r: any(kw in str(r).lower() for kw in keywords) if pd.notna(r) else False)
        mention_data.append(dict(study=label, condition=cond, hits=len(hits), mentions=int(mentions.sum()), rate=mentions.mean()))
mention_df = pd.DataFrame(mention_data)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(CONDS)-1); w = 0.35
conds_no_ctrl = [c for c in CONDS if c != 'control']
for i, (study, color) in enumerate([('Study 1', S1_COLOR), ('Study 2', S2_COLOR)]):
    sub = mention_df[mention_df['study']==study].set_index('condition').reindex(conds_no_ctrl)
    vals = sub['rate'].fillna(0).values
    bars = ax.bar(x + (i-0.5)*w, vals, w, label=study, color=color, alpha=0.85)
    for j, bar in enumerate(bars):
        if pd.notna(sub.iloc[j]['rate']):
            ax.text(bar.get_x()+bar.get_width()/2, vals[j]+0.02, f"{vals[j]:.0%}\n({int(sub.iloc[j]['mentions'])}/{int(sub.iloc[j]['hits'])})", ha='center', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([c.replace('_',' ').title() for c in conds_no_ctrl], fontsize=9)
ax.set_ylabel('Badge Mention Rate'); ax.set_title('Badge Mention Rate in Reasoning — Target Hits Only', fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.legend(fontsize=9); ax.set_ylim(0, 1.15)
plt.tight_layout(); plt.savefig('../results/260323_2/fig06_badge_mention.png', dpi=150, bbox_inches='tight'); plt.show()


## 11. Cost Summary


In [ ]:
for label, df in [("Study 1", s1), ("Study 2", s2)]:
    if len(df) == 0: continue
    cost = df['estimatedCostUsd'].sum(); time_sec = df['latencySec'].sum()
    print(f"{label}: {len(df):,} trials | ${cost:.2f} | {time_sec:,.0f}s ({time_sec/60:.1f}min) | {df['inputTokens'].sum():,.0f} in / {df['outputTokens'].sum():,.0f} out\n")


## 12. Export Summary


In [ ]:
summary = pd.DataFrame({
    'S1_rate': s1.groupby('condition')['choseTarget'].mean(),
    'S1_n': s1.groupby('condition')['choseTarget'].count(),
    'S2_rate': s2.groupby('condition')['choseTarget'].mean(),
    'S2_n': s2.groupby('condition')['choseTarget'].count(),
}).reindex(CONDS)
summary['S1_fmt'] = summary['S1_rate'].map(lambda x: f"{x:.1%}")
summary['S2_fmt'] = summary['S2_rate'].map(lambda x: f"{x:.1%}")
summary['delta'] = (summary['S2_rate'] - summary['S1_rate']).map(lambda x: f"{x:+.1%}")
print(summary[['S1_n','S1_fmt','S2_n','S2_fmt','delta']])
summary.to_csv('../results/260323_2/summary_by_condition.csv')
print('\nSaved: results/260323_2/summary_by_condition.csv')
